## Gold Baseline Modeling (Deliverable 1.3.2)

This notebook implements the baseline anomaly detection model using a single, broad Isolation Forest trained on the complete Gold feature set. The baseline serves as the primary comparison point for evaluating whether the three-stage cascade model improves alert quality.

**Purpose:**  
To produce the baseline model’s anomaly scores and alert outputs using the fully preprocessed Gold dataset. These outputs represent the simplest form of unsupervised anomaly detection and form the quantitative reference for the comparative evaluation in the Gold Comparison notebook.

**Key Goals:**

- Load the Gold preprocessed dataset and Stage 1 feature set.
- Train a single Isolation Forest using all vetted numeric features.
- Generate anomaly scores and binary alert flags.
- Produce baseline alert frequency counts, false-positive counts, and normal-period alert rates.
- Save all baseline model artifacts, metrics, and alert outputs for use in the Gold Comparison notebook.

**Relevance to Section C:**  
This notebook provides the baseline alert patterns used in Section C to evaluate whether the cascade reduces false positives, reduces noisy alerts, and preserves meaningful anomaly sensitivity. The outputs here form the “reference condition” for the paired statistical tests and practical significance analysis.

## Baseline Modeling Setup and Imports

In this section I am loading the libraries and project utilities needed for the Gold baseline modeling stage.

The purpose here is to get the notebook ready before I touch the Gold modeling data. That includes:
- standard Python libraries
- model and metrics utilities
- project path and config loading
- logging
- truth-record utilities
- artifact saving helpers
- experiment tracking support

At this point, I am not fitting the model yet. I am just setting up the notebook so the baseline workflow can run in a structured and repeatable way.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

from pathlib import Path
import yaml

import logging
import wandb

import pandas as pd 
import numpy as np 

import joblib 

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging, log_layer_paths

from utils.wandb_utils import finalize_wandb_stage

from utils.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.cascade_row_tracking import (
    ensure_stable_row_id,
    build_stage_scoring_frame,
    score_isolation_forest_stage,
    merge_stage_results_back,
    finalize_stage_flag_columns,
    get_detected_rows_dataframe,
    get_stage_detected_rows_dataframe,
)

from utils.postgres_util import get_engine_from_env
from utils.layer_postgres_writer import write_layer_dataframe, prepare_layer_dataframe


# Ledger 
from utils.ledger import Ledger


# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


----

## Load Configuration, Paths, and Baseline Runtime Settings

Here I load the resolved configuration values that control how the baseline notebook runs.

This step defines the key runtime pieces for the baseline stage, such as:
- dataset identity
- Gold version and recipe information
- threshold settings
- estimator count
- train fraction
- file names and artifact paths
- truth-store locations
- model output locations

I like doing this early because it keeps the rest of the notebook cleaner. Instead of scattering settings throughout the notebook, I define them once here and reuse them in the later steps.

In [2]:
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

CONFIG_ROOT = paths.configs
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage="gold_baseline",
    dataset="pump",
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

GOLD_CFG = CONFIG["gold_baseline"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]
PIPELINE = CONFIG.get(
    "pipeline",
    {
        "execution_mode": "batch",
        "orchestration_mode": "notebook",
    },
)

TRUTH_CONFIG = build_truth_config_block(CONFIG)
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

STAGE = "gold"
LAYER_NAME = GOLD_CFG["layer_name"]
GOLD_VERSION = CONFIG["versions"]["gold"]
RECIPE_ID = GOLD_CFG["recipe_id"]
TRUTH_VERSION = CONFIG["versions"]["truth"]

PIPELINE_MODE = PIPELINE["execution_mode"]
RUN_MODE = CONFIG["runtime"]["mode"]

DATASET_NAME_CONFIG = CONFIG["dataset"]["name"]
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

GOLD_PROCESS_RUN_ID = make_process_run_id(GOLD_CFG["process_run_id_prefix"])

# ---- W&B ----
WANDB_PROJECT = CONFIG["wandb"]["project"]
WANDB_ENTITY = CONFIG["wandb"]["entity"]
WANDB_RUN_NAME = f"{GOLD_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

GOLD_PREPROCESSED_FILE_NAME = FILENAMES["gold_preprocessed_file_name"]
GOLD_PREPROCESSED_SCALED_FILE_NAME = FILENAMES["gold_preprocessed_scaled_file_name"]
GOLD_FIT_FILE_NAME = FILENAMES["gold_fit_file_name"]
GOLD_TEST_FILE_NAME = FILENAMES["gold_test_file_name"]
GOLD_TRAIN_FILE_NAME = FILENAMES["gold_train_file_name"]

STAGE1_FEATURES_FILE_NAME = FILENAMES["stage1_features_file_name"]

BASELINE_RESULTS_FILE_NAME_CSV = FILENAMES["baseline_results_file_name_csv"]
BASELINE_RESULTS_FILE_NAME_PICKLE = FILENAMES["baseline_results_file_name_pickle"]
BASELINE_MODEL_FILE_NAME = FILENAMES["baseline_model_file_name"]
BASELINE_THRESHOLDS_FILE_NAME = FILENAMES["baseline_thresholds_file_name"]
BASELINE_SUMMARY_FILE_NAME = FILENAMES["baseline_summary_file_name"]
BASELINE_METADATA_FILE_NAME = FILENAMES["baseline_metadata_file_name"]

TRUTH_INDEX_FILE_NAME = "truth_index.jsonl"

TRAIN_FRACTION = float(GOLD_CFG["train_fraction"])
RANDOM_SEED = int(GOLD_CFG["random_seed"])
BASELINE_THRESHOLD_PERCENTILE = float(GOLD_CFG["baseline_threshold_percentile"])
STAGE1_THRESHOLD_PERCENTILE = float(GOLD_CFG["stage1_threshold_percentile"])
STAGE2_THRESHOLD_PERCENTILE = float(GOLD_CFG["stage2_threshold_percentile"])
BASELINE_ESTIMATOR_COUNT = int(GOLD_CFG["baseline_estimator_count"])

GOLD_BASELINE_LEDGER_FILE_NAME = FILENAMES["gold_baseline_ledger_file_name"]



# ---- Paths setup ----
GOLD_PREPROCESSED_DATA_PATH = Path(PATHS["gold_preprocessed_data_path"])
GOLD_PREPROCESSED_SCALED_DATA_PATH = Path(PATHS["gold_preprocessed_scaled_data_path"])

GOLD_TRAIN_DATA_PATH = Path(PATHS["gold_train_data_path"])
GOLD_TEST_DATA_PATH = Path(PATHS["gold_test_data_path"])
GOLD_FIT_DATA_PATH = Path(PATHS["gold_fit_data_path"])
GOLD_ARTIFACTS_PATH = Path(PATHS["gold_artifacts_dir"])

MODELS_PATH = Path(PATHS["models_root"])
BASELINE_MODELS_PATH = Path(PATHS["baseline_models_path"])
BASELINE_MODEL_ARTIFACT_PATH = Path(PATHS["baseline_model_artifact_path"])

BASELINE_RESULTS_PATH_CSV = Path(PATHS["baseline_results_path_csv"])
BASELINE_RESULTS_PATH_PICKLE = Path(PATHS["baseline_results_path_pickle"])
BASELINE_THRESHOLDS_PATH = Path(PATHS["baseline_thresholds_path"])
BASELINE_SUMMARY_PATH = Path(PATHS["baseline_summary_path"])
BASELINE_METADATA_PATH = Path(PATHS["baseline_metadata_path"])

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])

STAGE1_FEATURES_PATH = Path(PATHS["stage1_features_path"])

LOGS_PATH = Path(PATHS["logs_root"])

set_wandb_dir_from_config(CONFIG)

GOLD_PREPROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_PREPROCESSED_SCALED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_FIT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TEST_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_TRAIN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)


----

## Start Logging for the Baseline Modeling Stage

Before I load the modeling inputs, I want logging turned on so this notebook records what happened during the run.

This helps with debugging, lineage tracking, and basic pipeline discipline. If I need to confirm what inputs were used or where a step failed, the log gives me a cleaner record than notebook output alone.

In [3]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_modeling_baseline.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)


2026-04-05 06:31:59,464 | INFO | capstone.gold | Gold Modeling stage starting
2026-04-05 06:31:59,466 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-04-05 06:31:59,469 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-04-05 06:31:59,470 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-04-05 06:31:59,471 | INFO | capstone.gold | Project Notebooks Path Loaded: /workspace/notebooks
2026-04-05 06:31:59,473 | INFO | capstone.gold | Project Truths Path Loaded: /workspace/artifacts/truths
2026-04-05 06:31:59,474 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-04-05 06:31:59,475 | INFO | capstone.gold | Previous Layer (Silver) Path Loaded: /workspace/data/silver
2026-04-05 06:31:59,476 | INFO | capstone.gold | Previous Layer (Silver) Training Path Loaded: /workspace/data/silver/train
2026-04-05 06:31:59,478 | INFO | capstone.gold | Previous Layer (Silver) Testing Path Loaded: /workspace/data/s

----

## Initialize Experiment Tracking

This step starts the Weights & Biases run for the baseline modeling stage.

I am using this mainly for run tracking and artifact registration. It helps document the baseline configuration, input references, and saved outputs, but it does not change the underlying modeling logic itself.

In [4]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_modeling_baseline",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "baseline_threshold_percentile": BASELINE_THRESHOLD_PERCENTILE,
        "gold_input_path": str(GOLD_FIT_DATA_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dcoo230 (dcoo230-western-governors-university). Use `wandb login --relogin` to force relogin


2026-04-05 06:32:03,259 | INFO | capstone.gold | W&B initialized: gold__001


----

## Initialize the Baseline Ledger

Here I create the ledger that tracks the main steps taken during the baseline notebook.

I treat the ledger as a structured run record. It gives me a cleaner summary of the modeling workflow than relying only on printed notebook output, especially when I need to review or compare runs later.

In [5]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-04-05 06:32:03,584 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T06:32:03.584606+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-04-05T06:32:03.584606+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

----

## Load Gold Modeling Inputs and Resolve the Parent Truth

This is the point where I load the Gold preprocessing outputs that the baseline model depends on.

In this step I am:
- loading the scaled Gold dataframe
- resolving the parent Gold truth record
- confirming the dataset identity
- substituting truth-linked artifact paths when needed
- loading the Gold fit dataframe
- loading the Stage 1 feature list

This matters because the baseline notebook should not invent its own inputs. It should inherit the prepared modeling artifacts that were created in Gold preprocessing.

In [6]:
logger.info("Loading Gold Preprocessed parquet: %s", GOLD_PREPROCESSED_SCALED_DATA_PATH)
gold_preprocessed_scaled_dataframe = load_data(GOLD_PREPROCESSED_SCALED_DATA_PATH)

GOLD_DATASET_NAME = (
    gold_preprocessed_scaled_dataframe["meta__dataset"]
    .dropna()
    .astype("string")
    .str.strip()
)
GOLD_DATASET_NAME = GOLD_DATASET_NAME[GOLD_DATASET_NAME != ""]

if len(GOLD_DATASET_NAME) == 0:
    raise ValueError("Gold baseline input dataframe is missing usable meta__dataset values.")

GOLD_DATASET_NAME = str(GOLD_DATASET_NAME.iloc[0]).strip()

gold_truth = load_parent_truth_record_from_dataframe(
    dataframe=gold_preprocessed_scaled_dataframe,
    truth_dir=TRUTHS_PATH,
    parent_layer_name="gold",
    dataset_name=GOLD_DATASET_NAME,
    column_name="meta__truth_hash",
)

DATASET_NAME = get_dataset_name_from_truth(gold_truth)
GOLD_PARENT_TRUTH_HASH = get_truth_hash(gold_truth)

PARENT_PIPELINE_MODE = get_pipeline_mode_from_truth(gold_truth)
if PARENT_PIPELINE_MODE is not None:
    PIPELINE_MODE = PARENT_PIPELINE_MODE

GOLD_TRUTH_PATH = (
    TRUTHS_PATH
    / "gold"
    / f"{DATASET_NAME}__gold__truth__{GOLD_PARENT_TRUTH_HASH}.json"
)

gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

GOLD_FIT_DATA_PATH = Path(gold_truth_artifact_paths.get("gold_fit_path", str(GOLD_FIT_DATA_PATH)))
STAGE1_FEATURES_PATH = Path(gold_truth_artifact_paths.get("stage1_features_path", str(STAGE1_FEATURES_PATH)))

logger.info("Resolved Gold baseline dataset name from Gold truth: %s", DATASET_NAME)
logger.info("Resolved Gold truth path: %s", GOLD_TRUTH_PATH)
logger.info("Resolved Gold fit parquet from Gold truth: %s", GOLD_FIT_DATA_PATH)
logger.info("Resolved Stage 1 features path from Gold truth: %s", STAGE1_FEATURES_PATH)

logger.info("Loading Gold fit parquet: %s", GOLD_FIT_DATA_PATH)
gold_fit_dataframe = load_data(GOLD_FIT_DATA_PATH)

logger.info("Loading Stage 1 features JSON: %s", STAGE1_FEATURES_PATH)
stage1_feature_columns = load_json(STAGE1_FEATURES_PATH)

print("Gold baseline dataset name from parent truth:", DATASET_NAME)
print("Gold baseline parent truth hash:", GOLD_PARENT_TRUTH_HASH)

ledger.add(
    kind="step",
    step="load_modeling_inputs",
    message="Loaded Gold scaled parquet, loaded Gold truth, substituted truth-linked artifact paths, then loaded baseline inputs.",
    data={
        "gold_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "gold_fit_shape": list(gold_fit_dataframe.shape),
        "stage1_feature_count": int(len(stage1_feature_columns)),
    },
    logger=logger,
)

gold_fit_dataframe.head(3)

2026-04-05 06:32:03,895 | INFO | capstone.gold | Loading Gold Preprocessed parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-04-05 06:32:03,898 | INFO | capstone.file_io | Loading Parquet: /workspace/data/gold/pump__gold__preprocessed_scaled.parquet
2026-04-05 06:32:06,737 | INFO | capstone.gold | Resolved Gold baseline dataset name from Gold truth: pump
2026-04-05 06:32:06,741 | INFO | capstone.gold | Resolved Gold truth path: /workspace/artifacts/truths/gold/pump__gold__truth__d6c00152ed2f56139b4292f313a83aace2cfed3514a960c466053606a9f0be4e.json
2026-04-05 06:32:06,744 | INFO | capstone.gold | Resolved Gold fit parquet from Gold truth: /workspace/data/gold/pump__gold__fit_normal_only.parquet
2026-04-05 06:32:06,747 | INFO | capstone.gold | Resolved Stage 1 features path from Gold truth: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-04-05 06:32:06,750 | INFO | capstone.gold | Loading Gold fit parquet: /workspace/data/gold/pump__gold__fit

Gold baseline dataset name from parent truth: pump
Gold baseline parent truth hash: d6c00152ed2f56139b4292f313a83aace2cfed3514a960c466053606a9f0be4e


,meta__asset_id,meta__dataset,meta__episode_id,meta__event_id,meta__ingested_at_utc,meta__parent_truth_hash,meta__pipeline_mode,meta__record_id,meta__run_id,meta__source_file,meta__source_row_id,meta__split,meta__truth_hash,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status,meta__row_id,meta__is_train_flag
0,asset__001,pump,0,pump:asset__001:run__001:0,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,14598431322315673869,run__001,sensor.csv,0,fit_normal_only,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.232560,-0.509807,0.537036,1.090905,0.227271,-0.144184,-0.212771,0.000000,0.633343,-0.133358,-0.966540,0.663285,-0.099631,-0.152932,0.001485,-0.017980,0.259816,0.203007,0.012069,0.001266,0.026039,-0.004667,0.375041,0.470634,0.054448,0.018867,-0.398656,-0.746314,-0.137762,-0.554838,-1.070844,-0.832376,-0.653530,-0.038647,-0.157592,-0.223411,0.170945,-1.156251,-0.909090,0.527273,-0.909090,-0.882353,-0.125000,0.000000,4.000000,0.928571,-0.608697,0.715953,1.900001,0.014598,2018-04-01 00:00:00,NORMAL,0,True
1,asset__001,pump,0,pump:asset__001:run__001:1,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,15954729095895098000,run__001,sensor.csv,1,fit_normal_only,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,0.232560,-0.509807,0.537036,1.090905,0.227271,-0.144184,-0.212771,0.000000,0.633343,-0.133358,-0.966540,0.663285,-0.099631,-0.152932,0.001485,-0.017980,0.259816,0.203007,0.012069,0.001266,0.026039,-0.004667,0.375041,0.470634,0.054448,0.018867,-0.398656,-0.746314,-0.137762,-0.554838,-1.070844,-0.832376,-0.653530,-0.038647,-0.157592,-0.223411,0.170945,-1.156251,-0.909090,0.527273,-0.909090,-0.882353,-0.125000,0.000000,4.000000,0.928571,-0.608697,0.715953,1.900001,0.014598,2018-04-01 00:01:00,NORMAL,1,True
2,asset__001,pump,0,pump:asset__001:run__001:2,2026-04-05 01:27:10.767991+00:00,eb885b514594038c463c96ed2f58d75ad025af0b9588d9...,batch,10041703297090838359,run__001,sensor.csv,2,fit_normal_only,d6c00152ed2f56139b4292f313a83aace2cfed3514a960...,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,-0.255821,-0.392159,0.537036,1.127271,0.670453,-0.485083,-0.468102,-0.185694,0.750017,-0.333349,-0.869635,0.742744,0.084552,-0.140848,0.063680,0.024252,-0.023257,-0.051481,0.031807,0.039803,0.034031,0.045869,0.539582,0.559032,0.045512,0.031031,-0.029962,-0.769896,-0.025671,0.380643,-0.866485,-0.748481,-0.586075,-0.053782,-0.147948,-0.214093,0.276319,-1.031250,-0.954545,0.454546,-0.999999,-0.882353,-0.166667,-0.045454,3.954546,0.964285,-0.608696,0.688715,1.833334,0.072992,2018-04-01 00:02:00,NORMAL,2,True


----

## Validate Stable Row Identity for Baseline Scoring

Before baseline scoring begins, I want to confirm that the Gold preprocessing stage already stamped a stable row identifier that can be used to attach row-level anomaly outputs back to the full dataset.

In [7]:
gold_preprocessed_scaled_dataframe = ensure_stable_row_id(
    gold_preprocessed_scaled_dataframe,
    row_id_column="meta__row_id",
)

ledger.add(
    kind="step",
    step="validate_baseline_row_tracking",
    message="Validated stable row identity on Gold baseline modeling input dataframe.",
    data={
        "row_id_column": "meta__row_id",
        "row_count": int(len(gold_preprocessed_scaled_dataframe)),
        "row_id_unique": bool(gold_preprocessed_scaled_dataframe["meta__row_id"].is_unique),
    },
    logger=logger,
)

2026-04-05 06:32:08,857 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T06:32:08.857764+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'validate_baseline_row_tracking', 'message': 'Validated stable row identity on Gold baseline modeling input dataframe.', 'why': None, 'consequence': None, 'data': {'row_id_column': 'meta__row_id', 'row_count': 220320, 'row_id_unique': True}}


{'ts_utc': '2026-04-05T06:32:08.857764+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'validate_baseline_row_tracking',
 'message': 'Validated stable row identity on Gold baseline modeling input dataframe.',
 'why': None,
 'consequence': None,
 'data': {'row_id_column': 'meta__row_id',
  'row_count': 220320,
  'row_id_unique': True}}

----

## Build the Train and Test Masks

Before fitting or evaluating the model, I need to recover the train/test split that was already stamped during Gold preprocessing.

The key idea here is that the baseline notebook is **not** creating a new split. It is reusing the split created upstream so the modeling and comparison stages all work from the same partition.

In [8]:
# Masks (must exist in scaled parquet)
if "meta__is_train_flag" not in gold_preprocessed_scaled_dataframe.columns:
    raise ValueError("meta__is_train_flag missing from gold_preprocessed_scaled_dataframe. "
                     "Gold preprocessing must stamp it before saving.")


train_mask = gold_preprocessed_scaled_dataframe["meta__is_train_flag"].astype(bool)

test_mask = ~train_mask


logger.info("Split counts: all=%d train=%d test=%d", len(train_mask), int(train_mask.sum()), int(test_mask.sum()))
if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    logger.info("Anomalies: all=%d test=%d", 
                int(gold_preprocessed_scaled_dataframe["anomaly_flag"].fillna(0).astype(int).sum()),
                int(gold_preprocessed_scaled_dataframe.loc[test_mask, "anomaly_flag"].fillna(0).astype(int).sum()))

2026-04-05 06:32:09,201 | INFO | capstone.gold | Split counts: all=220320 train=136431 test=83889
2026-04-05 06:32:09,209 | INFO | capstone.gold | Anomalies: all=14484 test=118


### Ask

Why am I reusing the saved split instead of making a fresh split inside this notebook?

### Answer

I want the baseline results to stay aligned with the rest of the Gold workflow.

If this notebook made its own split, then the baseline, cascade, and comparison notebooks could end up evaluating different row partitions. That would make the comparison less reliable.

So this step is really about consistency. The split belongs to Gold preprocessing, and the model notebooks should reuse it.

----

## Prepare Labels and Feature Matrices

Now I prepare the arrays that the baseline model will use.

This includes:
- the available anomaly labels for evaluation
- the test-only labels
- the fit feature matrix built from the Gold fit dataframe
- the full feature matrix built from the scaled Gold dataframe

A detail that matters here is that the model is fit on the **fit-normal-only** subset, while scoring is done across the **full scaled Gold dataframe**. That means training and scoring are intentionally not using the exact same row set.

In [9]:
all_labels = None
test_labels = None

if "anomaly_flag" in gold_preprocessed_scaled_dataframe.columns:
    all_labels = gold_preprocessed_scaled_dataframe["anomaly_flag"].fillna(0).astype(int).values
    test_labels = gold_preprocessed_scaled_dataframe.loc[test_mask, "anomaly_flag"].fillna(0).astype(int).values
else:
    all_labels = None
    test_labels = None


baseline_train_fit_features = gold_fit_dataframe[stage1_feature_columns].values

baseline_all_features = gold_preprocessed_scaled_dataframe[stage1_feature_columns].values

#baseline_test_features = gold_test_dataframe[stage1_feature_columns].values

### Ask

Why am I fitting on the fit subset but scoring the full Gold dataset?

### Answer

Because those two steps are serving different purposes.

The fit subset is meant to represent the cleaner normal training behavior that the unsupervised model should learn from. The full Gold dataframe is what I want to evaluate and inspect after the model has been trained.

So the logic is:
- **fit on the normal reference behavior**
- **score the full dataset to see where alerts appear**
- **evaluate the scoring outcome on the held-out test rows**

----

## Define Baseline Scoring, Thresholding, and Evaluation Helpers

These helper functions break the baseline workflow into smaller pieces:
- computing anomaly scores from the Isolation Forest
- choosing a score threshold by percentile
- evaluating the thresholded predictions against available labels

I like separating these steps because it makes the notebook easier to read and easier to compare later when I move into the cascade notebooks.

In [10]:
def compute_anomaly_scores_isolation_forest(
    model: IsolationForest,
    feature_matrix: np.ndarray,
) -> np.ndarray:
    scores = model.score_samples(feature_matrix)
    anomaly_scores = -scores
    return anomaly_scores

In [11]:
def choose_threshold_by_percentile(
    anomaly_scores: np.ndarray,
    percentile: float,
) -> float:
    return float(np.percentile(anomaly_scores, percentile))

In [12]:
def evaluate_against_labels(
    true_labels: np.ndarray,
    anomaly_scores: np.ndarray,
    threshold: float,
) -> dict:
    predicted_labels = (anomaly_scores >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predicted_labels,
        average="binary",
        zero_division=0,
    )

    results = {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

    if len(np.unique(true_labels)) == 2:
        results["roc_auc"] = float(roc_auc_score(true_labels, anomaly_scores))
        results["pr_auc"] = float(average_precision_score(true_labels, anomaly_scores))
    else:
        results["roc_auc"] = None
        results["pr_auc"] = None

    return results

## Fit the Baseline Isolation Forest Model

This is the actual model-fitting step for the baseline notebook.

Here I initialize the Isolation Forest using the configured estimator count and random seed, then fit it on the Gold fit feature matrix. Since that fit matrix comes from the normal-only Gold fit dataframe, the model is being trained on the normal reference subset rather than on the full mixed dataset.

In [13]:
baseline_model = IsolationForest(
    n_estimators=BASELINE_ESTIMATOR_COUNT,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)


baseline_model.fit(baseline_train_fit_features)


IsolationForest(n_estimators=200, n_jobs=-1, random_state=42)

## Score the Dataset, Set the Threshold, and Build Baseline Results

After fitting the model, I score both:
- the normal-only fit subset
- the full scaled Gold dataframe

Then I use the training-score distribution to choose the anomaly threshold by percentile. After that, I apply the threshold to the full dataset scores and create the baseline results dataframe with:
- `baseline_score`
- `baseline_flag`

This is one of the most important steps in the notebook because it converts the fitted unsupervised model into row-level alert output that I can evaluate and save.

### Ask

Why am I choosing the threshold from the training-score distribution instead of from the test data?

### Answer

Because I do not want the test set deciding the baseline rule.

The threshold should come from the model's view of the normal training behavior, not from the held-out rows that I later use for evaluation. If I tuned the threshold on the test set, that would blur the line between training and evaluation.

So the logic here is:
- fit the model on the normal reference subset
- use that score distribution to choose the threshold
- apply the threshold to all rows
- evaluate the outcome on the test rows

In [14]:
# Score on Normal Only 
baseline_train_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_train_fit_features,
)

# Score on All Only 
baseline_all_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_all_features,
)


'''
# Score on Test
baseline_test_scores = compute_anomaly_scores_isolation_forest(
    baseline_model,
    baseline_test_features,
)
'''

if len(baseline_all_scores) != len(gold_preprocessed_scaled_dataframe):
    raise ValueError("Score length mismatch vs all-rows dataframe. Check feature matrix source.")

baseline_threshold = choose_threshold_by_percentile(
    baseline_train_scores,
    BASELINE_THRESHOLD_PERCENTILE,
)

#baseline_flags = (baseline_test_scores >= baseline_threshold).astype(int)
baseline_flags = (baseline_all_scores >= baseline_threshold).astype(int)



baseline_results = gold_preprocessed_scaled_dataframe.copy()



#baseline_results["baseline_score"] = baseline_test_scores

baseline_results["baseline_score"] = baseline_all_scores

baseline_results["baseline_flag"] = baseline_flags

baseline_metrics = {
    "model": "Baseline IsolationForest",
    "threshold_percentile": BASELINE_THRESHOLD_PERCENTILE,
    "threshold": float(baseline_threshold),
    "alert_count_all_rows": int(baseline_flags.sum()),
    "alert_count_test_rows": int(baseline_flags[test_mask.values].sum()),
}

if test_labels is not None:
    baseline_metrics.update(
        evaluate_against_labels(
            test_labels,
            #baseline_test_scores,
            baseline_all_scores[test_mask.values],
            baseline_threshold,
        )
    )

ledger.add(
    kind="step",
    step="run_baseline_isolation_forest",
    message="Ran baseline Isolation Forest fit on normal-only rows and scored the full scaled Gold dataset; evaluated on test rows.",
    data={
        "estimator_count": int(BASELINE_ESTIMATOR_COUNT),
        "threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
        "threshold": float(baseline_threshold),
        "training_rows": int(len(gold_fit_dataframe)),
        "test_rows": int(test_mask.sum()),
        "all_rows": int(len(gold_preprocessed_scaled_dataframe)),
        "train_rows": int(train_mask.sum()),
        "feature_count": int(len(stage1_feature_columns)),
        "alert_count_test_rows": int(baseline_flags[test_mask.values].sum()),
        "precision": baseline_metrics.get("precision"),
        "recall": baseline_metrics.get("recall"),
        "f1": baseline_metrics.get("f1"),
        "roc_auc": baseline_metrics.get("roc_auc"),
        "pr_auc": baseline_metrics.get("pr_auc"),
    },
    logger=logger,
)

baseline_metrics

2026-04-05 06:32:19,171 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T06:32:19.171536+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'run_baseline_isolation_forest', 'message': 'Ran baseline Isolation Forest fit on normal-only rows and scored the full scaled Gold dataset; evaluated on test rows.', 'why': None, 'consequence': None, 'data': {'estimator_count': 200, 'threshold_percentile': 97.0, 'threshold': 0.5524266654090823, 'training_rows': 122065, 'test_rows': 83889, 'all_rows': 220320, 'train_rows': 136431, 'feature_count': 50, 'alert_count_test_rows': 1522, 'precision': 0.04664914586070959, 'recall': 0.6016949152542372, 'f1': 0.08658536585365853, 'roc_auc': 0.9710702441624048, 'pr_auc': 0.4703307867606405}}


{'model': 'Baseline IsolationForest',
 'threshold_percentile': 97.0,
 'threshold': 0.5524266654090823,
 'alert_count_all_rows': 19493,
 'alert_count_test_rows': 1522,
 'precision': 0.04664914586070959,
 'recall': 0.6016949152542372,
 'f1': 0.08658536585365853,
 'roc_auc': 0.9710702441624048,
 'pr_auc': 0.4703307867606405}

### Ask

What should I pay attention to in the baseline metrics here?

### Answer

The main things I care about are:
- the chosen threshold percentile and actual threshold value
- how many alerts were raised across all rows
- how many alerts appeared in the test rows
- precision, recall, and F1 when test labels are available
- ROC AUC and PR AUC when both classes are present in the evaluation labels

This is the first benchmark for the project. So I am less worried about perfection here and more focused on creating a clean, traceable baseline that the cascade model can later be compared against.

----

In [16]:
baseline_feature_columns = list(stage1_feature_columns)

missing_baseline_feature_columns = [
    column_name
    for column_name in baseline_feature_columns
    if column_name not in gold_preprocessed_scaled_dataframe.columns
]

if missing_baseline_feature_columns:
    raise ValueError(
        "Some baseline feature columns are missing from gold_preprocessed_scaled_dataframe:\n"
        f"{missing_baseline_feature_columns[:25]}"
    )

baseline_stage_input_df = build_stage_scoring_frame(
    dataframe=gold_preprocessed_scaled_dataframe,
    feature_columns=baseline_feature_columns,
    mask=None,
    row_id_column="meta__row_id",
)

baseline_stage_results_df = score_isolation_forest_stage(
    stage_dataframe=baseline_stage_input_df,
    model=baseline_model,
    feature_columns=baseline_feature_columns,
    stage_name="baseline",
    row_id_column="meta__row_id",
)

gold_preprocessed_scaled_dataframe = merge_stage_results_back(
    master_dataframe=gold_preprocessed_scaled_dataframe,
    stage_results_dataframe=baseline_stage_results_df,
    stage_name="baseline",
    row_id_column="meta__row_id",
)

gold_preprocessed_scaled_dataframe = finalize_stage_flag_columns(
    gold_preprocessed_scaled_dataframe,
    stage_names=["baseline"],
)

baseline_all_scores = gold_preprocessed_scaled_dataframe["baseline_score"].to_numpy()
baseline_all_decisions = gold_preprocessed_scaled_dataframe["baseline_decision"].to_numpy()
baseline_all_preds = gold_preprocessed_scaled_dataframe["baseline_pred"].to_numpy()

ledger.add(
    kind="step",
    step="score_baseline_with_row_tracking",
    message="Scored the full Gold dataframe with baseline row-level tracking and merged outputs back by stable row id.",
    data={
        "row_count_scored": int(len(baseline_stage_results_df)),
        "feature_count": int(len(baseline_feature_columns)),
        "baseline_flag_count": int(gold_preprocessed_scaled_dataframe["baseline_flag"].sum()),
        "score_column": "baseline_score",
        "decision_column": "baseline_decision",
        "pred_column": "baseline_pred",
        "flag_column": "baseline_flag",
    },
    logger=logger,
)

/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
/opt/miniconda/envs/capstone/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
2026-04-05 06:37:13,605 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T06:37:13.605647+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'score_baseline_with_row_tracking', 'message': 'Scored the full Gold dataframe with baseline row-level tracking and merged outputs back by stable row id.', 'why': None, 'consequence': None, 'data': {'row_count_scored': 220320, 'feature_count': 50, 'baseline_flag_

{'ts_utc': '2026-04-05T06:37:13.605647+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'score_baseline_with_row_tracking',
 'message': 'Scored the full Gold dataframe with baseline row-level tracking and merged outputs back by stable row id.',
 'why': None,
 'consequence': None,
 'data': {'row_count_scored': 220320,
  'feature_count': 50,
  'baseline_flag_count': 35615,
  'score_column': 'baseline_score',
  'decision_column': 'baseline_decision',
  'pred_column': 'baseline_pred',
  'flag_column': 'baseline_flag'}}

----

## Build the Baseline Truth Record and Save the Baseline Artifacts

Now I turn the baseline results into a formal pipeline artifact.

This step does several important things:
- summarizes the baseline metrics
- builds the baseline truth record
- stamps lineage columns onto the result dataframe
- links the baseline output back to the parent Gold truth
- saves the results, model, thresholds, summary, metadata, and truth record

At this point, the baseline output becomes more than just notebook output. It becomes a tracked deliverable in the pipeline.

In [17]:
baseline_alert_count_all_rows = int(baseline_results["baseline_flag"].sum())
baseline_alert_count_test_rows = int(baseline_results.loc[test_mask, "baseline_flag"].sum())

baseline_thresholds = {
    "baseline_threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
    "baseline_threshold": float(baseline_threshold),
}

baseline_summary = {
    "dataset_name": DATASET_NAME,
    "baseline_metrics": baseline_metrics,
    "alert_count_all_rows": baseline_alert_count_all_rows,
    "alert_count_test_rows": baseline_alert_count_test_rows,
    "result_row_count": int(len(baseline_results)),
}

truth_config_snapshot = (
    TRUTH_CONFIG
    if "TRUTH_CONFIG" in globals()
    else {
        "runtime": {
            "stage": "gold_baseline",
            "dataset": DATASET_NAME,
            "mode": RUN_MODE if "RUN_MODE" in globals() else None,
            "profile": CONFIG_PROFILE if "CONFIG_PROFILE" in globals() else "default",
        }
    }
)

baseline_truth_layer_name = "gold_baseline"
baseline_process_run_id = (
    GOLD_PROCESS_RUN_ID
    if "GOLD_PROCESS_RUN_ID" in globals()
    else make_process_run_id("gold_baseline_process")
)

baseline_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=baseline_truth_layer_name,
    process_run_id=baseline_process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
)

baseline_truth = update_truth_section(
    baseline_truth,
    "config_snapshot",
    truth_config_snapshot,
)

baseline_truth = update_truth_section(
    baseline_truth,
    "runtime_facts",
    {
        "baseline_threshold_percentile": float(BASELINE_THRESHOLD_PERCENTILE),
        "baseline_threshold": float(baseline_threshold),
        "baseline_estimator_count": int(BASELINE_ESTIMATOR_COUNT),
        "train_fraction": float(TRAIN_FRACTION),
        "random_seed": int(RANDOM_SEED),
        "alert_count_all_rows": baseline_alert_count_all_rows,
        "alert_count_test_rows": baseline_alert_count_test_rows,
        "result_row_count": int(len(baseline_results)),
        "parent_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "gold_process_run_id": gold_truth.get("process_run_id"),
        "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    },
)

baseline_truth = update_truth_section(
    baseline_truth,
    "artifact_paths",
    {
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_preprocessed_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
        "gold_fit_path": str(GOLD_FIT_DATA_PATH),
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_models_path": str(BASELINE_MODELS_PATH),
        "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "baseline_metadata_path": str(BASELINE_METADATA_PATH),
    },
)

baseline_meta_columns = sorted(
    set(
        identify_meta_columns(baseline_results)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

baseline_feature_columns = identify_feature_columns(baseline_results)

baseline_truth_record = build_truth_record(
    truth_base=baseline_truth,
    row_count=len(baseline_results),
    column_count=baseline_results.shape[1] + 3,
    meta_columns=baseline_meta_columns,
    feature_columns=baseline_feature_columns,
)

BASELINE_TRUTH_HASH = baseline_truth_record["truth_hash"]

baseline_results = stamp_truth_columns(
    baseline_results,
    truth_hash=BASELINE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

baseline_truth_path = save_truth_record(
    baseline_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=baseline_truth_layer_name,
)

append_truth_index(
    baseline_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

baseline_summary["baseline_truth_hash"] = BASELINE_TRUTH_HASH
baseline_summary["baseline_truth_path"] = str(baseline_truth_path)
baseline_summary["baseline_process_run_id"] = baseline_process_run_id
baseline_summary["gold_truth_hash"] = GOLD_PARENT_TRUTH_HASH
baseline_summary["gold_truth_path"] = str(GOLD_TRUTH_PATH)
baseline_summary["gold_process_run_id"] = gold_truth.get("process_run_id")
baseline_summary["gold_feature_set_id"] = gold_truth_runtime_facts.get("feature_set_id")

baseline_metadata = {
    "gold_preprocessed_scaled_path": str(GOLD_PREPROCESSED_SCALED_DATA_PATH),
    "gold_fit_path": str(GOLD_FIT_DATA_PATH),
    "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
    "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
    "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
    "baseline_models_path": str(BASELINE_MODELS_PATH),
    "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
    "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
    "baseline_metadata_path": str(BASELINE_METADATA_PATH),

    # upstream truth linkage
    "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
    "gold_truth_path": str(GOLD_TRUTH_PATH),
    "gold_process_run_id": gold_truth.get("process_run_id"),
    "gold_feature_set_id": gold_truth_runtime_facts.get("feature_set_id"),
    "gold_scaler_path": gold_truth_artifact_paths.get("scaler_path"),
    "gold_scaler_kind": gold_truth_runtime_facts.get("scaler_kind_runtime"),
    "gold_recommended_imputation": gold_truth_runtime_facts.get("recommended_imputation"),

    # stage truth linkage
    "baseline_truth_hash": BASELINE_TRUTH_HASH,
    "baseline_truth_path": str(baseline_truth_path),
    "baseline_process_run_id": baseline_process_run_id,
}

baseline_results.to_csv(BASELINE_RESULTS_PATH_CSV, index=False)
baseline_results.to_pickle(BASELINE_RESULTS_PATH_PICKLE)


joblib.dump(baseline_model, BASELINE_MODEL_ARTIFACT_PATH)
joblib.dump(baseline_model, BASELINE_MODELS_PATH)

save_json(baseline_thresholds, BASELINE_THRESHOLDS_PATH)
save_json(baseline_summary, BASELINE_SUMMARY_PATH)
save_json(baseline_metadata, BASELINE_METADATA_PATH)

wandb.save(str(BASELINE_RESULTS_PATH_CSV))
wandb.save(str(BASELINE_RESULTS_PATH_PICKLE))
wandb.save(str(BASELINE_MODEL_ARTIFACT_PATH))
wandb.save(str(BASELINE_MODELS_PATH))
wandb.save(str(BASELINE_THRESHOLDS_PATH))
wandb.save(str(BASELINE_SUMMARY_PATH))
wandb.save(str(BASELINE_METADATA_PATH))
wandb.save(str(baseline_truth_path))

ledger.add(
    kind="step",
    step="save_baseline_outputs",
    message="Saved baseline results, trained Isolation Forest model, thresholds, summary, metadata, and baseline stage truth record.",
    data={
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_models_path": str(BASELINE_MODELS_PATH),
        "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "baseline_metadata_path": str(BASELINE_METADATA_PATH),
        "baseline_truth_hash": BASELINE_TRUTH_HASH,
        "baseline_truth_path": str(baseline_truth_path),
        "result_row_count": int(len(baseline_results)),
        "alert_count_all_rows": baseline_alert_count_all_rows,
        "alert_count_test_rows": baseline_alert_count_test_rows,
    },
    logger=logger,
)

2026-04-05 06:38:12,261 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__baseline_thresholds.json
2026-04-05 06:38:12,276 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__baseline_summary.json
2026-04-05 06:38:12,294 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__baseline_metadata.json
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
2026-04-05 06:38:12,659 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T06:38:12.659394+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'save_baseline_outputs', 'message': 'Saved baseline results, trained Isolation Forest model, thresholds, summary, metadata, and baseline stage truth record.', 'why': None, 'consequence': None, 'data': {'baseline_results_path_csv': '/workspace/artifac

{'ts_utc': '2026-04-05T06:38:12.659394+00:00',
 'stage': 'gold',
 'recipe': 'gold_modeling__v001_baseline',
 'kind': 'step',
 'step': 'save_baseline_outputs',
 'message': 'Saved baseline results, trained Isolation Forest model, thresholds, summary, metadata, and baseline stage truth record.',
 'why': None,
 'consequence': None,
 'data': {'baseline_results_path_csv': '/workspace/artifacts/gold/pump/pump__gold__baseline_results.csv',
  'baseline_results_path_pickle': '/workspace/artifacts/gold/pump/pump__gold__baseline_results.pkl',
  'baseline_model_artifact_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_isolation_forest.joblib',
  'baseline_models_path': '/workspace/models/pump/pump__gold__baseline_isolation_forest.joblib',
  'baseline_thresholds_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_thresholds.json',
  'baseline_summary_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_summary.json',
  'baseline_metadata_path': '/workspace/artifacts/gold/pu

----

In [18]:
baseline_detected_rows_dataframe = get_detected_rows_dataframe(
    dataframe=baseline_results,
    target_flag_column="baseline_flag",
    row_id_column="meta__row_id",
    score_column="baseline_score",
    decision_column="baseline_decision",
    pred_column="baseline_pred",
    include_columns=[
        "baseline_flag",
        "anomaly_flag",
    ],
    sort_by="time_index",
    ascending=True,
)

ledger.add(
    kind="step",
    step="extract_baseline_detected_rows",
    message="Built the baseline detected-rows dataframe from the scored baseline results using stable row tracking.",
    data={
        "target_flag_column": "baseline_flag",
        "row_id_column": "meta__row_id",
        "detected_row_count": int(len(baseline_detected_rows_dataframe)),
        "has_time_index": bool("time_index" in baseline_detected_rows_dataframe.columns),
        "has_event_step": bool("event_step" in baseline_detected_rows_dataframe.columns),
        "has_event_time": bool("event_time" in baseline_detected_rows_dataframe.columns),
    },
    logger=logger,
)

print(f"Baseline detected row count: {len(baseline_detected_rows_dataframe):,}")
display(baseline_detected_rows_dataframe.head(20))

2026-04-05 06:38:13,423 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T06:38:13.423146+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'extract_baseline_detected_rows', 'message': 'Built the baseline detected-rows dataframe from the scored baseline results using stable row tracking.', 'why': None, 'consequence': None, 'data': {'target_flag_column': 'baseline_flag', 'row_id_column': 'meta__row_id', 'detected_row_count': 19493, 'has_time_index': True, 'has_event_step': True, 'has_event_time': True}}


Baseline detected row count: 19,493


,meta__row_id,meta__record_id,time_index,event_step,event_time,meta__asset_id,meta__run_id,machine_status,anomaly_flag,baseline_flag,baseline_score
0,17154,11031252919804255274,17154,17154,2018-04-12 21:54:00+00:00,asset__001,run__001,NORMAL,0,1,0.568583
1,17155,7306247312672969352,17155,17155,2018-04-12 21:55:00+00:00,asset__001,run__001,BROKEN,1,1,0.577982
2,17156,16171677129378469200,17156,17156,2018-04-12 21:56:00+00:00,asset__001,run__001,RECOVERING,1,1,0.574671
3,17157,14211091896624741348,17157,17157,2018-04-12 21:57:00+00:00,asset__001,run__001,RECOVERING,1,1,0.598155
4,17158,313266691668351730,17158,17158,2018-04-12 21:58:00+00:00,asset__001,run__001,RECOVERING,1,1,0.579311
5,17159,5086261863442700928,17159,17159,2018-04-12 21:59:00+00:00,asset__001,run__001,RECOVERING,1,1,0.582706
6,17160,11141286012755120792,17160,17160,2018-04-12 22:00:00+00:00,asset__001,run__001,RECOVERING,1,1,0.583654
7,17161,12435410761567819601,17161,17161,2018-04-12 22:01:00+00:00,asset__001,run__001,RECOVERING,1,1,0.583654
8,17162,16834369260743897814,17162,17162,2018-04-12 22:02:00+00:00,asset__001,run__001,RECOVERING,1,1,0.581123
9,17163,13012502951683690117,17163,17163,2018-04-12 22:03:00+00:00,asset__001,run__001,RECOVERING,1,1,0.600989


----

## Finalize the Ledger and Close the Tracking Run

This step writes the baseline ledger to disk and cleanly closes the experiment tracking run.

By the time I get here, the important modeling work and artifact creation are already complete. So this section is mainly about wrapping up the notebook in a structured way.

In [19]:
ledger.add(
    kind="step",
    step="finalize_baseline_modeling",
    message="Gold baseline modeling notebook complete.",
    data={
        "baseline_metrics": baseline_metrics,
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_model_artifact_path": str(BASELINE_MODEL_ARTIFACT_PATH),
        "baseline_model_path": str(BASELINE_MODELS_PATH),
    },
    logger=logger,
)

baseline_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_BASELINE_LEDGER_FILE_NAME
ledger.write_json(baseline_ledger_path)

wandb.save(str(baseline_ledger_path))
wandb_run.finish()

2026-04-05 06:38:13,968 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-04-05T06:38:13.968153+00:00', 'stage': 'gold', 'recipe': 'gold_modeling__v001_baseline', 'kind': 'step', 'step': 'finalize_baseline_modeling', 'message': 'Gold baseline modeling notebook complete.', 'why': None, 'consequence': None, 'data': {'baseline_metrics': {'model': 'Baseline IsolationForest', 'threshold_percentile': 97.0, 'threshold': 0.5524266654090823, 'alert_count_all_rows': 19493, 'alert_count_test_rows': 1522, 'precision': 0.04664914586070959, 'recall': 0.6016949152542372, 'f1': 0.08658536585365853, 'roc_auc': 0.9710702441624048, 'pr_auc': 0.4703307867606405}, 'baseline_results_path_csv': '/workspace/artifacts/gold/pump/pump__gold__baseline_results.csv', 'baseline_results_path_pickle': '/workspace/artifacts/gold/pump/pump__gold__baseline_results.pkl', 'baseline_model_artifact_path': '/workspace/artifacts/gold/pump/pump__gold__baseline_isolation_forest.joblib', 'baseline_model_path': '/workspace/models

----

## Run Final Lineage and Consistency Checks

Before I treat the baseline notebook as complete, I run a final sanity check on the saved baseline results and truth artifacts.

This check verifies things like:
- required lineage columns exist in the results dataframe
- the dataframe truth hash matches the saved baseline truth
- the parent truth hash matches the Gold parent truth
- the saved truth file exists
- the saved metadata points back to the correct baseline truth hash

I like ending with this because it confirms that the baseline output is not only saved, but also internally consistent and traceable.

In [20]:
required_baseline_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_baseline_meta_columns = [
    column_name
    for column_name in required_baseline_meta_columns
    if column_name not in baseline_results.columns
]
if missing_baseline_meta_columns:
    raise ValueError(
        f"baseline_results is missing required lineage columns: {missing_baseline_meta_columns}"
    )

baseline_results_truth_hash_check = extract_truth_hash(baseline_results)
if baseline_results_truth_hash_check is None:
    raise ValueError("baseline_results does not contain a readable meta__truth_hash value.")

if baseline_results_truth_hash_check != BASELINE_TRUTH_HASH:
    raise ValueError(
        "baseline_results truth hash does not match BASELINE_TRUTH_HASH:\n"
        f"dataframe={baseline_results_truth_hash_check}\n"
        f"record={BASELINE_TRUTH_HASH}"
    )

baseline_parent_values = baseline_results["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
if not baseline_parent_values:
    raise ValueError("baseline_results is missing populated meta__parent_truth_hash values.")

if len(baseline_parent_values) != 1:
    raise ValueError(f"baseline_results has multiple parent truth hashes: {baseline_parent_values}")

if baseline_parent_values[0] != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "baseline_results parent truth hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"dataframe_parent={baseline_parent_values[0]}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

if not Path(baseline_truth_path).exists():
    raise FileNotFoundError(f"Baseline truth file was not created: {baseline_truth_path}")

loaded_baseline_truth = load_json(baseline_truth_path)

if loaded_baseline_truth.get("truth_hash") != BASELINE_TRUTH_HASH:
    raise ValueError(
        "Saved Baseline truth file hash does not match BASELINE_TRUTH_HASH:\n"
        f"file={loaded_baseline_truth.get('truth_hash')}\n"
        f"record={BASELINE_TRUTH_HASH}"
    )

if loaded_baseline_truth.get("parent_truth_hash") != GOLD_PARENT_TRUTH_HASH:
    raise ValueError(
        "Saved Baseline truth file parent hash does not match GOLD_PARENT_TRUTH_HASH:\n"
        f"truth={loaded_baseline_truth.get('parent_truth_hash')}\n"
        f"gold_parent={GOLD_PARENT_TRUTH_HASH}"
    )

saved_baseline_metadata = load_json(BASELINE_METADATA_PATH)
if saved_baseline_metadata.get("baseline_truth_hash") != BASELINE_TRUTH_HASH:
    raise ValueError(
        "baseline_metadata baseline_truth_hash does not match BASELINE_TRUTH_HASH:\n"
        f"metadata={saved_baseline_metadata.get('baseline_truth_hash')}\n"
        f"record={BASELINE_TRUTH_HASH}"
    )

print("Gold Baseline lineage sanity check passed.")

2026-04-05 06:39:06,602 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/gold_baseline/pump__gold_baseline__truth__23cfdf317382ede4f2e9a130d2b3a7b6c3aa59f96b394c4e4e22de3c9f70c644.json
2026-04-05 06:39:06,617 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/gold/pump/pump__gold__baseline_metadata.json


Gold Baseline lineage sanity check passed.


### Ask

What does this final sanity check tell me?

### Answer

It tells me whether the baseline results can actually be trusted as a pipeline artifact.

A notebook can run successfully and still produce outputs with broken lineage or mismatched truth metadata. This final check helps guard against that by making sure the dataframe, truth record, and saved metadata all agree with each other.

So this is really a trust check, not just a completion check.

----

## Optional PostgreSQL Write for Baseline Outputs

This section is for writing the baseline results into PostgreSQL.

I am treating this as an optional persistence step rather than part of the core baseline modeling deliverable. The main outputs from this notebook are still the saved baseline results, trained model artifact, thresholds, summary, metadata, ledger, and truth record. Writing to SQL is useful when I want the baseline artifact available for querying, validation, or downstream integration work.

SQL_SCHEMAS = {
    "bronze": "bronze",
    "silver": "silver",
    "gold": "gold",
    "synthetic": "synthetic",
    "truth": "truth",
    "audit": "audit",
}

GOLD_ARTIFACT_TABLES = {
    "baseline_results": "baseline_results",
    "baseline_summary": "baseline_summary",
    "baseline_thresholds": "baseline_thresholds",
    "cascade_results": "cascade_results",
    "cascade_summary": "cascade_summary",
    "cascade_thresholds": "cascade_thresholds",
    "comparison_summary": "comparison_summary",
}

SYNTHETIC_ARTIFACT_TABLES = {
    "batch": "batch",
    "stream": "stream",
    "sensor_messages": "sensor_messages",
}





GOLD_SCHEMA = SQL_SCHEMAS["gold"]

engine = get_engine_from_env()



gold_baseline_results_sql = prepare_layer_dataframe(
    baseline_results,
    truth_hash=BASELINE_TRUTH_HASH,
    parent_truth_hash=GOLD_PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
    process_run_id=GOLD_PROCESS_RUN_ID,
    add_loaded_at_column=True,
)

gold_baseline_results_table = write_layer_dataframe(
    engine=engine,
    dataframe=gold_baseline_results_sql,
    schema=GOLD_SCHEMA,
    dataset_name=DATASET_NAME,
    artifact_name=GOLD_ARTIFACT_TABLES["baseline_results"],
    if_exists="replace",
    index=False,
)

print(f"Wrote table: {GOLD_SCHEMA}.{gold_baseline_results_table}")